# Femur Round 2: Modern Techniques Benchmark

Evaluates modern techniques on FP+UCL Femur test set (N=348 images / 266 patients):
- **Row A**: Baseline U-Net EffNet-B4 1-ch heatmap (384px, standard peak decode, flip TTA)
- **Row R1**: HRNet-W32 heatmap (384px, DARK sub-pixel decode, flip TTA)
- **Row R2**: Row-A net with integral soft-argmax coordinate loss, decode by soft-argmax
- **Row R3**: Row-A net with auxiliary bone line heatmap + length loss + min_dist=50px decoding
- **Row R4**: Row-A net + strong ultrasound-style augmentation
- **Row R5**: Pretrained encoder check (DINOv2 / UltraSam) - documented and skipped
- **Row R6**: Median ensemble of Row A and top 2 rows by validation MAE only

Final femur recipe selected strictly by validation MAE only.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

results_dir = Path('/kaggle/working/out')
if not results_dir.is_dir():
    results_dir = Path('../../docs/results/femur_round2')
if not results_dir.is_dir():
    results_dir = Path('docs/results/femur_round2')

table_path = results_dir / 'femur_round2_table.md'
if table_path.is_file():
    display(Markdown(table_path.read_text(encoding='utf-8')))
else:
    print('Table not found at', table_path)

# Femur Round 2: Modern Techniques (FP + UCL Femur Test Set, N=348 images / 266 patients)

> Reference FL: Euclidean distance between annotated endpoints * px_to_mm_rate.
> Protocol: Train on 451 images, early stop on 65 validation images (0 patient overlap).
> Evaluation: 348 official test images (323 FP / 261 patients, 25 UCL / 5 patients [tiny]).
> Paired Diff: Patient-level bootstrap with 1,000 resamples of difference in FL MAE vs Row A.
> Selection: Best recipe chosen strictly by validation MAE only.

| Approach | Row / Model | Resolution | Val MAE (mm) | Epochs | Train Time (s) | FL MAE [95% CI] (mm) | Paired Diff vs Row A [95% CI] (mm) | FL RMSE [95% CI] (mm) | FL Median [95% CI] (mm) | % Error [95% CI] | FL NME (mean+-SD) | Top-10 SqErr Share (%) | FP MAE (mm) | UCL MAE (mm, tiny) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| Row A (Baseline EffNet-B4 1-ch Heatmap, flip TTA) | *Seeds Mean +- SD* | 384 px | 1.150 | 14.3 +- 2.5 | 252.4 +- 34.9 | 1.314 +- 0.162 | - | 3.256 +- 0.777 | 0.704 +- 0.137 | 3.29% +- 0.51% | 0.039 +- 0.003 | 81.8% +- 7.9% | 1.230 | 2.397 |
| Row A (Baseline EffNet-B4 1-ch Heatmap, flip TTA) | **3-Seed Median Ens** | 384 px | 1.150 | - | - | **1.350** [1.065, 1.692] | Reference | **3.786** [1.799, 5.723] | **0.667** [0.597, 0.786] | **3.46%** [2.48%, 4.85%] | **0.038 +- 0.151** | **88.7%** | **1.227** | **2.937** |
| Row R1 (HRNet-W32 Heatmap, DARK Sub-pixel) | *Seeds Mean +- SD* | 384 px | 16.863 | 33.7 +- 1.9 | 338.8 +- 26.5 | 18.213 +- 0.898 | - | 25.911 +- 0.706 | 12.946 +- 1.875 | 40.31% +- 2.55% | 0.424 +- 0.009 | 18.4% +- 0.8% | 18.453 | 15.117 |
| Row R1 (HRNet-W32 Heatmap, DARK Sub-pixel) | **3-Seed Median Ens** | 384 px | 16.863 | - | - | **18.333** [16.119, 20.535] | **16.983** [14.770, 19.191] | **26.143** [23.712, 28.328] | **13.618** [9.349, 16.827] | **40.20%** [36.07%, 44.04%] | **0.415 +- 0.389** | **17.8%** | **18.863** | **11.489** |
| Row R2 (Row-A Net, Integral Soft-Argmax Loss) | *Seeds Mean +- SD* | 384 px | 1.220 | 30.0 +- 3.7 | 512.7 +- 64.0 | 1.223 +- 0.079 | - | 2.351 +- 0.347 | 0.645 +- 0.049 | 2.91% +- 0.26% | 0.043 +- 0.001 | 63.3% +- 6.5% | 1.179 | 1.794 |
| Row R2 (Row-A Net, Integral Soft-Argmax Loss) | **3-Seed Median Ens** | 384 px | 1.220 | - | - | **1.071** [0.911, 1.262] | **-0.279** [-0.640, 0.007] *(no measurable difference)* | **2.017** [1.522, 2.540] | **0.550** [0.493, 0.685] | **2.49%** [2.10%, 2.96%] | **0.040 +- 0.167** | **63.6%** | **1.097** | **0.742** |
| Row R3 (Row-A Net, Bone Line + Length Loss, MinDist=50px) | *Seeds Mean +- SD* | 384 px | 28.293 | 10.7 +- 1.7 | 183.4 +- 29.4 | 43.204 +- 15.958 | - | 50.121 +- 17.762 | 41.482 +- 16.670 | 104.88% +- 39.20% | 1.159 +- 0.304 | 15.9% +- 0.1% | 43.073 | 44.894 |
| Row R3 (Row-A Net, Bone Line + Length Loss, MinDist=50px) | **3-Seed Median Ens** | 384 px | 28.293 | - | - | **47.505** [44.820, 50.539] | **46.155** [43.247, 49.214] | **54.760** [51.881, 57.986] | **46.231** [40.249, 49.928] | **118.92%** [110.00%, 128.88%] | **1.062 +- 0.505** | **15.6%** | **47.713** | **44.809** |
| ★ Row R4 (Row-A Net + Strong Ultrasound Aug) | *Seeds Mean +- SD* | 384 px | 1.070 | 21.0 +- 5.7 | 367.5 +- 100.1 | 1.129 +- 0.090 | - | 3.577 +- 0.197 | 0.553 +- 0.038 | 2.93% +- 0.25% | 0.037 +- 0.003 | 91.0% +- 0.4% | 0.995 | 2.869 |
| ★ Row R4 (Row-A Net + Strong Ultrasound Aug) | **3-Seed Median Ens** | 384 px | 1.070 | - | - | **1.118** [0.856, 1.459] | **-0.232** [-0.398, -0.086] | **3.546** [1.471, 5.507] | **0.538** [0.459, 0.604] | **2.90%** [1.96%, 4.23%] | **0.033 +- 0.141** | **91.1%** | **0.980** | **2.901** |
| Row R6 (Ensemble of Row A + Best 2 Val Rows) | **9-Model Multi-Row Ens** | 384 px | 1.147 | - | - | **1.196** [0.919, 1.537] | **-0.154** [-0.272, -0.038] | **3.590** [1.600, 5.522] | **0.578** [0.500, 0.654] | **3.08%** [2.12%, 4.42%] | **0.036 +- 0.147** | **89.8%** | **1.065** | **2.883** |

### Pretrained Encoder Check (Row R5):
- **DINOv2 ViT-S/14**: Apache 2.0 license, publicly downloadable weights from Facebook Research hub. Incompatible with standard 384px resolution because patch size is 14 (`384 % 14 != 0`), violating the standardized letterbox resolution protocol. Skipped.
- **UltraSam (arXiv 2411.16222)**: Derivative of SAM requiring external heavy checkpoint hosting and 1024px architecture. Skipped due to lack of standard PyTorch distribution.


In [2]:
json_path = results_dir / 'femur_round2_results.json'
if json_path.is_file():
    with open(json_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    print('Loaded round 2 results for keys:', list(results.keys()))
    for rk in ['row_A', 'row_R1', 'row_R2', 'row_R3', 'row_R4', 'row_R6']:
        if rk in results:
            ens = results[rk]['ensemble_all']
            ms = results[rk]['seeds_mean_sd']
            paired = results[rk].get('paired_vs_row_A', {})
            print(f'=== {results[rk]["name"]} ===')
            print(f'  Val MAE:    {ms.get("val_mae_mean", float("nan")):.3f} mm')
            print(f'  Ensemble:   MAE={ens["fl_mae"]:.3f} [{ens["fl_mae_ci"][0]:.3f}, {ens["fl_mae_ci"][1]:.3f}] mm, NME={ens["fl_nme_mean"]:.3f} +- {ens["fl_nme_sd"]:.3f}')
            if rk != 'row_A':
                diff = paired.get('diff_mae', 0.0)
                ci = [paired.get('ci_low', 0.0), paired.get('ci_high', 0.0)]
                meas = ' (no measurable difference)' if paired.get('ci_includes_zero') else ''
                print(f'  Paired vs A: {diff:.3f} [{ci[0]:.3f}, {ci[1]:.3f}] mm{meas}')
else:
    print('Results JSON not found at', json_path)

Loaded round 2 results for keys: ['row_A', 'row_R1', 'row_R2', 'row_R3', 'row_R4', 'row_R6']
=== Row A (EffNet-B4 1-ch Heatmap, 384px) ===
  Val MAE:    1.150 mm
  Ensemble:   MAE=1.350 [1.065, 1.692] mm, NME=0.038 +- 0.151
=== Row R1 (HRNet-W32 Heatmap, DARK Decode, 384px) ===
  Val MAE:    16.863 mm
  Ensemble:   MAE=18.333 [16.119, 20.535] mm, NME=0.415 +- 0.389
  Paired vs A: 16.983 [14.770, 19.191] mm
=== Row R2 (Row-A Net, Integral Soft-Argmax, 384px) ===
  Val MAE:    1.220 mm
  Ensemble:   MAE=1.071 [0.911, 1.262] mm, NME=0.040 +- 0.167
  Paired vs A: -0.279 [-0.640, 0.007] mm (no measurable difference)
=== Row R3 (Row-A Net, Bone Line + Length Loss, 384px) ===
  Val MAE:    28.293 mm
  Ensemble:   MAE=47.505 [44.820, 50.539] mm, NME=1.062 +- 0.505
  Paired vs A: 46.155 [43.247, 49.214] mm
=== Row R4 (Row-A Net + Strong Ultrasound Aug, 384px) ===
  Val MAE:    1.070 mm
  Ensemble:   MAE=1.118 [0.856, 1.459] mm, NME=0.033 +- 0.141
  Paired vs A: -0.232 [-0.398, -0.086] mm
=== Ro

In [3]:
csv_path = results_dir / 'femur_round2_predictions.csv'
if csv_path.is_file():
    df = pd.read_csv(csv_path)
    print(f'Total predictions: {len(df)} images across {df["patient"].nunique()} patients')
    assert len(df) == 348, f'Expected 348 test images, got {len(df)}'
    print('Verification check passed: 348 test predictions confirmed.')
else:
    print('Predictions CSV not found at', csv_path)

Total predictions: 348 images across 266 patients
Verification check passed: 348 test predictions confirmed.
